# 04. Prophet 모델

## 목표
- Prophet 기본 피팅 + 체인지포인트 시각화
- 외생 regressor (유가, 공휴일, 프로모션) 추가 비교
- 하이퍼파라미터 튜닝 (changepoint_prior_scale, seasonality_prior_scale, seasonality_mode)
- Prophet 내장 Cross-Validation
- Validation 예측 + 신뢰구간 + 베이스라인/SARIMA 대비 비교

In [1]:
import sys

sys.path.insert(0, "..")

import os
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
import logging

logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

fm._load_fontmanager(try_read_cache=False)
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

from src.models.prophet_model import ProphetModel
from src.evaluation import mape, evaluate_model, compare_models

FIG_DIR = os.path.abspath(
    os.path.join(os.path.dirname("__file__"), "..", "outputs", "figures")
)
RES_DIR = os.path.abspath(
    os.path.join(os.path.dirname("__file__"), "..", "outputs", "results")
)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

print("Setup 완료")

Setup 완료


In [2]:
full = pd.read_csv("../data/processed/full_data.csv", parse_dates=["date"])
train = pd.read_csv("../data/processed/train.csv", parse_dates=["date"])
val = pd.read_csv("../data/processed/val.csv", parse_dates=["date"])
baseline = pd.read_csv("../outputs/results/baseline_results.csv")
sarima = pd.read_csv("../outputs/results/sarima_results.csv")

families = full["family"].unique().tolist()
REGRESSORS = ["oil_price", "is_holiday", "onpromotion"]

print(f"Train: {train.shape}, Val: {val.shape}")
print(f"상품군: {families}")
print(f"Regressors: {REGRESSORS}")

Train: (7285, 19), Val: (905, 19)
상품군: ['BEVERAGES', 'CLEANING', 'DAIRY', 'GROCERY I', 'PRODUCE']
Regressors: ['oil_price', 'is_holiday', 'onpromotion']


---
## 1. Prophet 기본 피팅 + 체인지포인트

각 상품군에 대해 기본 Prophet 모델을 피팅하고:
- 자동 감지된 체인지포인트 시각화
- 트렌드 / 주간 계절성 / 연간 계절성 분해

In [3]:
fig, axes = plt.subplots(len(families), 2, figsize=(18, 4 * len(families)))

basic_models = {}
for i, family in enumerate(families):
    m = ProphetModel()
    m.fit(train, family)
    basic_models[family] = m

    # 컴포넌트 예측
    comp = m.get_components()

    # 트렌드 + 체인지포인트
    axes[i, 0].plot(comp["ds"], comp["trend"], color="#e74c3c", linewidth=1.5)
    # 체인지포인트 표시
    if hasattr(m.model_, "changepoints"):
        for cp in m.model_.changepoints:
            axes[i, 0].axvline(
                x=cp, color="gray", linewidth=0.5, alpha=0.5, linestyle="--"
            )
    axes[i, 0].set_title(f"{family} - 트렌드 + 체인지포인트", fontsize=10)
    axes[i, 0].set_ylabel("트렌드")

    # 주간 계절성
    weekly = comp[["ds", "weekly"]].copy()
    weekly["dow"] = weekly["ds"].dt.dayofweek
    weekly_avg = weekly.groupby("dow")["weekly"].mean()
    days = ["월", "화", "수", "목", "금", "토", "일"]
    axes[i, 1].bar(range(7), weekly_avg.values, color="#3498db", alpha=0.7)
    axes[i, 1].set_xticks(range(7))
    axes[i, 1].set_xticklabels(days)
    axes[i, 1].set_title(f"{family} - 주간 계절성", fontsize=10)
    axes[i, 1].axhline(y=0, color="red", linewidth=0.5)

plt.suptitle("Prophet 트렌드 + 계절성 분해", fontsize=13, fontweight="bold")
plt.tight_layout()
save_path = os.path.join(FIG_DIR, "15_prophet_components.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {save_path}")

23:28:30 - cmdstanpy - INFO - Chain [1] start processing


23:28:30 - cmdstanpy - INFO - Chain [1] done processing


23:28:31 - cmdstanpy - INFO - Chain [1] start processing


23:28:31 - cmdstanpy - INFO - Chain [1] done processing


23:28:31 - cmdstanpy - INFO - Chain [1] start processing


...

23:28:31 - cmdstanpy - INFO - Chain [1] start processing


23:28:31 - cmdstanpy - INFO - Chain [1] done processing


23:28:32 - cmdstanpy - INFO - Chain [1] start processing


23:28:32 - cmdstanpy - INFO - Chain [1] done processing


저장: E:\프로젝트\부족한 프로젝트 시작\수요예측 시스템\outputs\figures\15_prophet_components.png


---
## 2. Prophet Basic vs Prophet + Regressors

외생변수(oil_price, is_holiday, onpromotion) 추가 효과를 비교한다.

In [4]:
prophet_models_basic = {}
prophet_models_reg = {}
all_results = []

for family in families:
    val_sub = val[val["family"] == family].sort_values("date")
    steps = len(val_sub)
    y_true = val_sub["sales"].values

    print(f"\n{family}")

    # --- Prophet Basic ---
    m1 = ProphetModel()
    m1.fit(train, family)
    pred1 = m1.predict(periods=steps)
    y_pred1 = pred1["forecast"].values[: len(y_true)]

    r1 = evaluate_model(
        y_true,
        y_pred1,
        "Prophet_Basic",
        family,
        train_time=m1.train_time_,
        predict_time=m1.predict_time_,
    )
    all_results.append(r1)
    prophet_models_basic[family] = m1
    print(
        f"  Basic:      MAPE={r1['mape']:.2f}%, RMSE={r1['rmse']:.0f}, 학습={m1.train_time_:.1f}초"
    )

    # --- Prophet + Regressors ---
    m2 = ProphetModel(regressors=REGRESSORS)
    m2.fit(train, family)
    pred2 = m2.predict(periods=steps, val_df=val)
    y_pred2 = pred2["forecast"].values[: len(y_true)]

    r2 = evaluate_model(
        y_true,
        y_pred2,
        "Prophet_Reg",
        family,
        train_time=m2.train_time_,
        predict_time=m2.predict_time_,
    )
    all_results.append(r2)
    prophet_models_reg[family] = m2
    print(
        f"  Regressors: MAPE={r2['mape']:.2f}%, RMSE={r2['rmse']:.0f}, 학습={m2.train_time_:.1f}초"
    )

prophet_df = compare_models(all_results)
print("\n모델 학습 완료")

23:28:33 - cmdstanpy - INFO - Chain [1] start processing


23:28:33 - cmdstanpy - INFO - Chain [1] done processing



BEVERAGES


23:28:33 - cmdstanpy - INFO - Chain [1] start processing


  Basic:      MAPE=46.96%, RMSE=45342, 학습=0.1초


...

23:28:35 - cmdstanpy - INFO - Chain [1] done processing


  Regressors: MAPE=91.01%, RMSE=52374, 학습=0.1초

PRODUCE
  Basic:      MAPE=70.67%, RMSE=38426, 학습=0.1초


23:28:35 - cmdstanpy - INFO - Chain [1] start processing


23:28:35 - cmdstanpy - INFO - Chain [1] done processing


  Regressors: MAPE=69.68%, RMSE=38637, 학습=0.1초

모델 학습 완료


In [5]:
print("Prophet Basic vs Regressors 비교:")
print(
    prophet_df[["model", "family", "mape", "rmse", "mae", "train_time_sec"]].to_string(
        index=False, float_format="{:.2f}".format
    )
)

print("\n상품군별 최적 모델 (MAPE 기준):")
best_per_family = prophet_df.loc[prophet_df.groupby("family")["mape"].idxmin()]
for _, row in best_per_family.iterrows():
    print(f"  {row['family']:15s} → {row['model']} (MAPE={row['mape']:.2f}%)")

Prophet Basic vs Regressors 비교:
        model    family   mape     rmse      mae  train_time_sec
Prophet_Basic BEVERAGES  46.96 45341.89 32803.58            0.13
  Prophet_Reg BEVERAGES  58.78 73701.32 65792.36            0.17
  Prophet_Reg  CLEANING 124.38 13529.49  8792.88            0.15
Prophet_Basic  CLEANING 131.66 12947.63  8092.76            0.13
Prophet_Basic     DAIRY  68.12  8266.84  5173.19            0.12
  Prophet_Reg     DAIRY  70.13  8395.64  5408.22            0.14
  Prophet_Reg GROCERY I  91.01 52373.88 39037.26            0.13
Prophet_Basic GROCERY I 100.35 43277.56 25681.11            0.13
  Prophet_Reg   PRODUCE  69.68 38637.35 31823.82            0.12
Prophet_Basic   PRODUCE  70.67 38425.69 32147.50            0.12

상품군별 최적 모델 (MAPE 기준):
  BEVERAGES       → Prophet_Basic (MAPE=46.96%)
  CLEANING        → Prophet_Reg (MAPE=124.38%)
  DAIRY           → Prophet_Basic (MAPE=68.12%)
  GROCERY I       → Prophet_Reg (MAPE=91.01%)
  PRODUCE         → Prophet_Reg (MAPE=69.

---
## 3. 하이퍼파라미터 튜닝

주요 파라미터 Grid Search:
- `changepoint_prior_scale`: [0.001, 0.01, 0.05, 0.1, 0.5]
- `seasonality_prior_scale`: [0.1, 1.0, 10.0]
- `seasonality_mode`: ['additive', 'multiplicative']

Validation MAPE 기준으로 최적 조합을 선택한다.

In [6]:
# 탐색 공간 (시간 절약을 위해 핵심 조합)
param_grid = [
    {"cps": 0.001, "sps": 10.0, "mode": "additive"},
    {"cps": 0.01, "sps": 10.0, "mode": "additive"},
    {"cps": 0.05, "sps": 10.0, "mode": "additive"},
    {"cps": 0.1, "sps": 10.0, "mode": "additive"},
    {"cps": 0.5, "sps": 10.0, "mode": "additive"},
    {"cps": 0.05, "sps": 0.1, "mode": "additive"},
    {"cps": 0.05, "sps": 1.0, "mode": "additive"},
    {"cps": 0.05, "sps": 10.0, "mode": "multiplicative"},
    {"cps": 0.1, "sps": 1.0, "mode": "multiplicative"},
]

tuning_results = []
best_params = {}

print("하이퍼파라미터 튜닝 (9개 조합)...")
print("=" * 70)

for family in families:
    val_sub = val[val["family"] == family].sort_values("date")
    y_true = val_sub["sales"].values
    steps = len(val_sub)

    best_mape = np.inf
    best_p = None

    for params in param_grid:
        try:
            m = ProphetModel(
                changepoint_prior_scale=params["cps"],
                seasonality_prior_scale=params["sps"],
                seasonality_mode=params["mode"],
                regressors=REGRESSORS,
            )
            m.fit(train, family)
            pred = m.predict(periods=steps, val_df=val)
            y_pred = pred["forecast"].values[: len(y_true)]
            m_mape = mape(y_true, y_pred)

            tuning_results.append(
                {
                    "family": family,
                    "cps": params["cps"],
                    "sps": params["sps"],
                    "mode": params["mode"],
                    "mape": m_mape,
                }
            )

            if m_mape < best_mape:
                best_mape = m_mape
                best_p = params.copy()
                best_p["mape"] = m_mape
        except Exception:
            continue

    best_params[family] = best_p
    print(
        f"{family:15s} → cps={best_p['cps']}, sps={best_p['sps']}, "
        f"mode={best_p['mode']}, MAPE={best_p['mape']:.2f}%"
    )

print("=" * 70)
print("튜닝 완료")

23:28:35 - cmdstanpy - INFO - Chain [1] start processing


23:28:35 - cmdstanpy - INFO - Chain [1] done processing


하이퍼파라미터 튜닝 (9개 조합)...


23:28:35 - cmdstanpy - INFO - Chain [1] start processing


23:28:35 - cmdstanpy - INFO - Chain [1] done processing


...

23:28:45 - cmdstanpy - INFO - Chain [1] start processing


23:28:45 - cmdstanpy - INFO - Chain [1] done processing


23:28:45 - cmdstanpy - INFO - Chain [1] start processing


23:28:45 - cmdstanpy - INFO - Chain [1] done processing


PRODUCE         → cps=0.5, sps=10.0, mode=additive, MAPE=59.99%
튜닝 완료


---
## 4. 최적 파라미터로 최종 모델 학습

In [7]:
tuned_models = {}
tuned_results = []

for family in families:
    bp = best_params[family]
    val_sub = val[val["family"] == family].sort_values("date")
    y_true = val_sub["sales"].values
    steps = len(val_sub)

    m = ProphetModel(
        changepoint_prior_scale=bp["cps"],
        seasonality_prior_scale=bp["sps"],
        seasonality_mode=bp["mode"],
        regressors=REGRESSORS,
    )
    m.fit(train, family)
    pred = m.predict(periods=steps, val_df=val)
    y_pred = pred["forecast"].values[: len(y_true)]

    r = evaluate_model(
        y_true,
        y_pred,
        "Prophet_Tuned",
        family,
        train_time=m.train_time_,
        predict_time=m.predict_time_,
    )
    tuned_results.append(r)
    tuned_models[family] = m

    print(
        f"{family:15s} → MAPE={r['mape']:.2f}%, RMSE={r['rmse']:.0f}, "
        f"cps={bp['cps']}, sps={bp['sps']}, mode={bp['mode']}"
    )

# 전체 결과 합산
final_results = all_results + tuned_results
final_df = compare_models(final_results)
save_path = os.path.join(RES_DIR, "prophet_results.csv")
final_df.to_csv(save_path, index=False)
print(f"\n저장: {save_path}")

23:28:45 - cmdstanpy - INFO - Chain [1] start processing


23:28:46 - cmdstanpy - INFO - Chain [1] done processing


BEVERAGES       → MAPE=43.24%, RMSE=41838, cps=0.01, sps=10.0, mode=additive

23:28:46 - cmdstanpy - INFO - Chain [1] start processing


23:28:46 - cmdstanpy - INFO - Chain [1] done processing


...

DAIRY           → MAPE=67.25%, RMSE=8158, cps=0.1, sps=10.0, mode=additive


23:28:46 - cmdstanpy - INFO - Chain [1] start processing


GROCERY I       → MAPE=90.82%, RMSE=54012, cps=0.1, sps=10.0, mode=additive


23:28:47 - cmdstanpy - INFO - Chain [1] done processing


PRODUCE         → MAPE=59.99%, RMSE=22902, cps=0.5, sps=10.0, mode=additive

저장: E:\프로젝트\부족한 프로젝트 시작\수요예측 시스템\outputs\results\prophet_results.csv


---
## 5. Cross-Validation

Prophet 내장 CV로 모델의 시간에 따른 예측 성능을 평가한다.
- initial: 730일 (2년 학습)
- period: 180일마다 CV
- horizon: 181일 (6개월 예측)

In [8]:
cv_all = {}
print("Cross-Validation 실행...")

for family in families:
    m = tuned_models[family]
    try:
        cv_results, perf = m.cross_validate(
            initial="730 days",
            period="180 days",
            horizon="181 days",
        )
        cv_all[family] = {"cv": cv_results, "perf": perf}
        mean_mape = perf["mape"].mean() * 100
        print(f"  {family:15s} → CV 평균 MAPE={mean_mape:.2f}%")
    except Exception as e:
        print(f"  {family:15s} → CV 실패: {e}")
        cv_all[family] = None

print("CV 완료")

Cross-Validation 실행...


  0%|          | 0/4 [00:00<?, ?it/s]

23:28:47 - cmdstanpy - INFO - Chain [1] start processing


23:28:47 - cmdstanpy - INFO - Chain [1] done processing


23:28:47 - cmdstanpy - INFO - Chain [1] start processing


...

23:28:51 - cmdstanpy - INFO - Chain [1] start processing


23:28:51 - cmdstanpy - INFO - Chain [1] done processing


23:28:51 - cmdstanpy - INFO - Chain [1] start processing


23:28:51 - cmdstanpy - INFO - Chain [1] done processing


  PRODUCE         → CV 평균 MAPE=7378.19%
CV 완료


In [9]:
# CV horizon별 MAPE 시각화
fig, axes = plt.subplots(len(families), 1, figsize=(14, 3 * len(families)), sharex=True)

for i, family in enumerate(families):
    if cv_all[family] is None:
        axes[i].set_title(f"{family} - CV 실패", fontsize=10)
        continue

    perf = cv_all[family]["perf"]
    horizon_days = perf["horizon"].dt.days
    axes[i].plot(horizon_days, perf["mape"] * 100, color="#e74c3c", linewidth=1.5)
    axes[i].fill_between(
        horizon_days, 0, perf["mape"] * 100, alpha=0.1, color="#e74c3c"
    )
    axes[i].set_title(f"{family}", fontsize=10, fontweight="bold")
    axes[i].set_ylabel("MAPE (%)")
    axes[i].axhline(y=50, color="gray", linewidth=0.5, linestyle="--", alpha=0.5)

axes[-1].set_xlabel("예측 기간 (일)")
plt.suptitle("Prophet Cross-Validation: Horizon별 MAPE", fontsize=13, fontweight="bold")
plt.tight_layout()
save_path = os.path.join(FIG_DIR, "17_prophet_cv.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {save_path}")

저장: E:\프로젝트\부족한 프로젝트 시작\수요예측 시스템\outputs\figures\17_prophet_cv.png


---
## 6. 예측 vs 실제 + 신뢰구간

In [10]:
fig, axes = plt.subplots(
    len(families), 1, figsize=(16, 3.5 * len(families)), sharex=True
)

for i, family in enumerate(families):
    model = tuned_models[family]
    val_sub = val[val["family"] == family].sort_values("date")
    steps = len(val_sub)

    pred = model.predict(periods=steps, val_df=val)
    y_true = val_sub["sales"].values[: len(pred)]
    dates = val_sub["date"].values[: len(pred)]

    # 80% CI (Prophet의 interval_width 조정)
    model_80 = ProphetModel(
        changepoint_prior_scale=best_params[family]["cps"],
        seasonality_prior_scale=best_params[family]["sps"],
        seasonality_mode=best_params[family]["mode"],
        regressors=REGRESSORS,
    )
    model_80.model_ = model.model_
    model_80.model_.interval_width = 0.80
    model_80.family_ = family
    pred_80 = model_80.predict(periods=steps, val_df=val)

    # Train 마지막 30일
    train_tail = train[train["family"] == family].sort_values("date").tail(30)

    axes[i].plot(
        train_tail["date"],
        train_tail["sales"],
        color="gray",
        linewidth=1,
        label="Train",
    )
    axes[i].plot(dates, y_true, color="black", linewidth=1.2, label="실제")
    axes[i].plot(
        dates,
        pred["forecast"].values[: len(y_true)],
        color="#2ecc71",
        linewidth=1.2,
        label="Prophet 예측",
    )
    axes[i].fill_between(
        dates,
        pred["lower_ci"].values[: len(y_true)],
        pred["upper_ci"].values[: len(y_true)],
        alpha=0.1,
        color="#2ecc71",
        label="95% CI",
    )
    axes[i].fill_between(
        dates,
        pred_80["lower_ci"].values[: len(y_true)],
        pred_80["upper_ci"].values[: len(y_true)],
        alpha=0.2,
        color="#2ecc71",
        label="80% CI",
    )
    axes[i].set_title(family, fontsize=11, fontweight="bold")
    axes[i].legend(loc="upper left", fontsize=8)

plt.suptitle("Prophet 예측 vs 실제 (Validation 기간)", fontsize=13, fontweight="bold")
plt.tight_layout()
save_path = os.path.join(FIG_DIR, "16_prophet_forecast.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {save_path}")

저장: E:\프로젝트\부족한 프로젝트 시작\수요예측 시스템\outputs\figures\16_prophet_forecast.png


---
## 7. 베이스라인 / SARIMA / Prophet 3종 비교

In [11]:
# 각 모델의 상품군별 최적 결과
best_baseline = baseline.loc[baseline.groupby("family")["mape"].idxmin()]
best_sarima = sarima.loc[sarima.groupby("family")["mape"].idxmin()]
best_prophet = final_df.loc[final_df.groupby("family")["mape"].idxmin()]

comparison = []
for family in families:
    bl = best_baseline[best_baseline["family"] == family].iloc[0]
    sr = best_sarima[best_sarima["family"] == family].iloc[0]
    pr = best_prophet[best_prophet["family"] == family].iloc[0]

    comparison.append(
        {
            "상품군": family,
            "베이스라인": f"{bl['model']} ({bl['mape']:.1f}%)",
            "SARIMA": f"{sr['model']} ({sr['mape']:.1f}%)",
            "Prophet": f"{pr['model']} ({pr['mape']:.1f}%)",
            "최적": min(
                [
                    ("베이스라인", bl["mape"]),
                    ("SARIMA", sr["mape"]),
                    ("Prophet", pr["mape"]),
                ],
                key=lambda x: x[1],
            )[0],
        }
    )

comp_df = pd.DataFrame(comparison)
print("3종 모델 비교 (베이스라인 vs SARIMA vs Prophet):")
print(comp_df.to_string(index=False))

3종 모델 비교 (베이스라인 vs SARIMA vs Prophet):
      상품군            베이스라인           SARIMA                Prophet      최적
BEVERAGES Naive_7d (44.0%)  SARIMAX (49.7%)  Prophet_Tuned (43.2%) Prophet
 CLEANING  MA_30d (120.0%) SARIMAX (139.8%) Prophet_Tuned (124.2%)   베이스라인
    DAIRY   MA_30d (65.8%)  SARIMAX (89.5%)  Prophet_Tuned (67.3%)   베이스라인
GROCERY I    MA_7d (98.1%)  SARIMAX (65.2%)  Prophet_Tuned (90.8%)  SARIMA
  PRODUCE   MA_30d (62.1%)  SARIMAX (79.8%)  Prophet_Tuned (60.0%) Prophet


---
## 8. 요약 및 인사이트

In [12]:
print("=" * 60)
print("Day 4 Prophet 분석 요약")
print("=" * 60)
print()
print("1. 최적 하이퍼파라미터:")
for family in families:
    bp = best_params[family]
    print(f"   {family:15s} → cps={bp['cps']}, sps={bp['sps']}, mode={bp['mode']}")
print()
print("2. Prophet 최적 MAPE:")
for _, row in best_prophet.iterrows():
    print(f"   {row['family']:15s} → {row['model']} MAPE={row['mape']:.2f}%")
print()
avg_prophet = best_prophet["mape"].mean()
avg_sarima = best_sarima["mape"].mean()
avg_baseline = best_baseline["mape"].mean()
print("3. 전체 평균 MAPE:")
print(f"   베이스라인: {avg_baseline:.1f}%")
print(f"   SARIMA:    {avg_sarima:.1f}%")
print(f"   Prophet:   {avg_prophet:.1f}%")
print()
print("4. 인사이트:")
print("   - Prophet은 체인지포인트 자동 감지로 트렌드 변화에 강함")
print("   - 외생변수(유가, 공휴일, 프로모션) 추가 시 성능 변화 확인")
print("   - CV 결과: 예측 기간이 길어질수록 MAPE 증가")
print("   - 다음 단계: Day 5 (XGBoost + 최종 모델 비교)")

Day 4 Prophet 분석 요약

1. 최적 하이퍼파라미터:
   BEVERAGES       → cps=0.01, sps=10.0, mode=additive
   CLEANING        → cps=0.05, sps=0.1, mode=additive
   DAIRY           → cps=0.1, sps=10.0, mode=additive
   GROCERY I       → cps=0.1, sps=10.0, mode=additive
   PRODUCE         → cps=0.5, sps=10.0, mode=additive

2. Prophet 최적 MAPE:
   BEVERAGES       → Prophet_Tuned MAPE=43.24%
   CLEANING        → Prophet_Tuned MAPE=124.18%
   DAIRY           → Prophet_Tuned MAPE=67.25%
   GROCERY I       → Prophet_Tuned MAPE=90.82%
   PRODUCE         → Prophet_Tuned MAPE=59.99%

3. 전체 평균 MAPE:
   베이스라인: 78.0%
   SARIMA:    84.8%
   Prophet:   77.1%

4. 인사이트:
   - Prophet은 체인지포인트 자동 감지로 트렌드 변화에 강함
   - 외생변수(유가, 공휴일, 프로모션) 추가 시 성능 변화 확인
   - CV 결과: 예측 기간이 길어질수록 MAPE 증가
   - 다음 단계: Day 5 (XGBoost + 최종 모델 비교)
